# Criação da Tabela Fato — Assistência Estudantil (UFPB, Campus I)

Este notebook constrói a **tabela fato** de assistência estudantil a partir
dos dados do SEDAP+ (Censo da Educação Superior 2024), já usando as chaves
das dimensões geradas em `1_criacao_dimensoes.ipynb` / `src/build_dimensions.py`.

**Colunas finais da fato:**
`ID_CURSO`, `ID_SEXO`, `ID_RACA`, `ID_TURNO`, `TP_LEI_COTAS`,
`IN_ACAO_AFIRMATIVA`, `IN_APOIO_SOCIAL`, `IN_APOIO_ALIMENTACAO`,
`IN_APOIO_MORADIA`, `IN_APOIO_TRANSPORTE`, `IN_APOIO_MATERIAL_DIDATICO`,
`IN_APOIO_BOLSA_PERMANENCIA`, `IN_APOIO_BOLSA_TRABALHO`, `TOTAL_ALUNOS`,
`RECEBE_AUXILIO`, `IDPNA`, `TOTAL_IDPNA`

**Etapas do notebook:**
1. Importação das bibliotecas
2. Leitura da tabela fato
3. Filtrar Campus I
4. Renomear colunas
5. Atualizar a dimensão turno (aplicar na fato)
6. Tratar valores nulos
7. Criar `RECEBE_AUXILIO`
8. Validar `IN_ACAO_AFIRMATIVA`
9. Criar `IDPNA`
10. Criar `TOTAL_IDPNA`
11. Validações
12. Exportação do CSV final

## 1. Importação das bibliotecas

In [ ]:
import pandas as pd

## 2. Leitura da tabela fato

Leitura dos dados brutos do SEDAP+, ainda sem nenhum filtro ou tratamento.
A extração já vem com a coluna `IN_ACAO_AFIRMATIVA` (ver etapa 8).

In [ ]:
fato = pd.read_csv("../data/raw/FATO_ACAO.csv")

## 3. Filtrar Campus I

O SEDAP+ não permite filtrar o Campus I na consulta SQL. O filtro é feito
aqui em Python, removendo os cursos que não pertencem ao Campus I através da
mesma lista de códigos (`cursos_fora`) usada na `dim_curso`
(`src/build_dimensions.py`).

O código `113701` (Ciências Agrárias — Licenciatura EaD) foi incluído nessa
lista: embora apareça nos dados de João Pessoa, o curso pertence
administrativamente ao CCHSA (Campus III — Bananeiras) e não tem vínculo
presencial com nenhum dos 13 centros do Campus I. Mantê-lo na fato inflaria
indevidamente os números do Campus I.

In [ ]:
cursos_fora = [
    13403, 13454, 13455, 13457, 80589, 97767, 98976, 98980,
    98982, 98984, 99045, 107348, 107352, 107356, 107360,
    109626, 113709, 397767, 1161324, 1167933, 1440696,
    5000897, 5000898, 113699, 1110415,
    113701  # Ciências Agrárias EaD — CCHSA / Campus III, fora do escopo
]

fato = fato[
    ~fato["CO_CURSO"].isin(cursos_fora)
]

In [ ]:
print("Cursos após filtro de campus:", fato["CO_CURSO"].nunique())

## 4. Renomear colunas

Renomeia as colunas de origem do SEDAP+ para os nomes de chave usados nas
dimensões (`ID_CURSO`, `ID_SEXO`, `ID_RACA`, `ID_TURNO`).

In [ ]:
fato = fato.rename(columns={
    "CO_CURSO": "ID_CURSO",
    "TP_SEXO": "ID_SEXO",
    "TP_COR_RACA": "ID_RACA",
    "TP_TURNO": "ID_TURNO"
})

## 5. Atualizar a dimensão turno (aplicar na fato)

Existem cursos **EaD sem turno preenchido** (`ID_TURNO` nulo). Esses
registros **não devem ser excluídos**: substituímos o valor nulo por
`ID_TURNO = 0`, que corresponde ao registro `"Não informado"` adicionado na
`dim_turno` (`src/build_dimensions.py`, função `build_dim_turno`).

In [ ]:
fato["ID_TURNO"] = (
    fato["ID_TURNO"]
    .fillna(0)
    .astype(int)
)

## 6. Tratar valores nulos

Foi identificado o seguinte padrão nas colunas de auxílio:
- `IN_APOIO_SOCIAL = 0` → os demais auxílios vêm nulos (não se aplicam).
- `IN_APOIO_SOCIAL = 1` → os auxílios específicos vêm preenchidos com 0 ou 1.

Por isso:
- **Não** preenchemos `IN_APOIO_SOCIAL`.
- Preenchemos com `0` apenas os nulos das colunas de auxílio específico:
  `IN_APOIO_ALIMENTACAO`, `IN_APOIO_MORADIA`, `IN_APOIO_TRANSPORTE`,
  `IN_APOIO_MATERIAL_DIDATICO`, `IN_APOIO_BOLSA_PERMANENCIA`,
  `IN_APOIO_BOLSA_TRABALHO`.

In [ ]:
# Diagnóstico: quantidade de nulos antes do tratamento
fato[
    [
        "IN_APOIO_SOCIAL",
        "IN_APOIO_ALIMENTACAO",
        "IN_APOIO_MORADIA",
        "IN_APOIO_TRANSPORTE",
        "IN_APOIO_MATERIAL_DIDATICO",
        "IN_APOIO_BOLSA_PERMANENCIA",
        "IN_APOIO_BOLSA_TRABALHO"
    ]
].isnull().sum()

In [ ]:
# Preenche com 0 apenas as colunas de auxílio específico (NÃO mexe em IN_APOIO_SOCIAL)
colunas_auxilio = [
    "IN_APOIO_ALIMENTACAO",
    "IN_APOIO_MORADIA",
    "IN_APOIO_TRANSPORTE",
    "IN_APOIO_MATERIAL_DIDATICO",
    "IN_APOIO_BOLSA_PERMANENCIA",
    "IN_APOIO_BOLSA_TRABALHO"
]

fato[colunas_auxilio] = (
    fato[colunas_auxilio]
    .fillna(0)
    .astype(int)
)

## 7. Criar RECEBE_AUXILIO

`RECEBE_AUXILIO` é o indicador unificado de recebimento de algum auxílio —
uma cópia (como inteiro) de `IN_APOIO_SOCIAL`.

In [ ]:
fato["RECEBE_AUXILIO"] = fato["IN_APOIO_SOCIAL"].astype(int)

## 8. Validar `IN_ACAO_AFIRMATIVA`

A coluna `IN_RESERVA_VAGAS` está zerada/nula no Censo 2024 e por isso foi
substituída por `IN_ACAO_AFIRMATIVA` para identificar o ingresso por ação
afirmativa:

- **Cobertura mais ampla que `TP_LEI_COTAS`:** `IN_ACAO_AFIRMATIVA` engloba
  tanto os cotistas da Lei Federal nº 12.711/2012 (renda, raça, escola
  pública) quanto os ingressantes por cotas/bonificações **institucionais**
  próprias da UFPB — casos que `TP_LEI_COTAS` sozinho não capturava.
- `TP_LEI_COTAS` é mantido na fato como coluna informativa (detalha o tipo
  específico de cota da Lei 12.711 quando aplicável), mas **não** é mais a
  variável usada para calcular o `IDPNA` (etapa 9).

Antes de usar `IN_ACAO_AFIRMATIVA` no cálculo do IDPNA, validamos que a
coluna é binária (0/1) e sem nulos inesperados.

In [ ]:
print(fato["IN_ACAO_AFIRMATIVA"].value_counts(dropna=False))

# Caso existam nulos, tratamos como "não ingressou por ação afirmativa" (0)
fato["IN_ACAO_AFIRMATIVA"] = (
    fato["IN_ACAO_AFIRMATIVA"]
    .fillna(0)
    .astype(int)
)

## 9. Criar IDPNA

`IDPNA` (indicador de aluno ingressante por ação afirmativa e **não**
assistido) identifica os alunos que:
- ingressaram por ação afirmativa (`IN_ACAO_AFIRMATIVA == 1`); **e**
- não recebem nenhum auxílio (`RECEBE_AUXILIO == 0`).

In [ ]:
fato["IDPNA"] = (
    (fato["IN_ACAO_AFIRMATIVA"] == 1) &
    (fato["RECEBE_AUXILIO"] == 0)
).astype(int)

## 10. Criar TOTAL_IDPNA

Total de alunos "IDPNA" por linha, considerando `TOTAL_ALUNOS` (o número de
alunos representado por aquela combinação de dimensões).

In [ ]:
fato["TOTAL_IDPNA"] = (
    fato["IDPNA"] *
    fato["TOTAL_ALUNOS"]
)

## 11. Validações

Conferências finais antes de exportar: amostra dos dados, colunas, nulos
restantes, quantidade de cursos, distribuições de `IDPNA` /
`RECEBE_AUXILIO`, total de alunos e comparação entre os cursos da
`dim_curso_completo` (processada em `src/build_dimensions.py`) e os cursos
presentes na fato.

In [ ]:
display(fato.head(10))

In [ ]:
print(fato.columns)

In [ ]:
fato.isnull().sum()

In [ ]:
print("Cursos:", fato["ID_CURSO"].nunique())

In [ ]:
print(fato["IDPNA"].value_counts())

In [ ]:
print(fato["RECEBE_AUXILIO"].value_counts())

In [ ]:
print("Total de alunos:", fato["TOTAL_ALUNOS"].sum())

In [ ]:
fato.groupby("ID_CURSO")[["TOTAL_ALUNOS", "TOTAL_IDPNA"]].sum().head(10)

In [ ]:
# Confere se há algum curso presente na dim_curso (Campus I) que não aparece na fato
dim_curso_check = pd.read_csv("../data/processed/dim_curso_completo.csv")

cursos_dim = set(dim_curso_check["ID_CURSO"])
cursos_fato = set(fato["ID_CURSO"])

faltando = cursos_dim - cursos_fato

print("Quantidade:", len(faltando))
print(sorted(faltando))

In [ ]:
dim_curso_check[
    dim_curso_check["ID_CURSO"].isin(faltando)
][["ID_CURSO", "CURSO"]]

## 12. Exportação do CSV final

Exportação da fato **já tratada** (nulos tratados, `RECEBE_AUXILIO`,
`IDPNA` e `TOTAL_IDPNA` calculados) — pronta para uso no dashboard
Streamlit.

In [ ]:
fato.to_csv(
    "../data/processed/fato_final.csv",
    index=False,
    encoding="utf-8-sig"
)